# Random-path laser weld: neural-operator temperature inference

Ground-truth (FLOW-3D) vs. neural-operator prediction of the temperature field for
**arbitrary laser weld paths** (random + sinusoidal), in the lab frame with time
inference. Three operators are compared: **FNO**, **DeepONet**, **RIGNO**.

The model is conditioned on the laser trajectory via explicit input channels derived
from each case's `case_manifest.json` (instantaneous beam centre `beam_x/beam_y`, a
Gaussian beam footprint `beam_dist`, and an accumulated exposure `path_dose` that
carries the path history). No moving reference frame is used.

**Viewer:** top = ground truth, bottom = prediction, for a surface-normal (x-y) slice.
Independent sliders for **time** and for **z** (depth); a case/model selector; the
laser path and instantaneous beam centre are overlaid.

Run the eval entrypoint first for each model, then point `MODEL_DIRS` at the eval run
dirs (the ones containing `fno_eval/predictions.npz`).

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

REPO = Path.cwd().resolve().parents[1]  # notebook lives in src/notebooks/
TEST_DIR = REPO / "data/flow3D/random_paths/test"
TIMESTEPS_PER_CASE = 20     # frames per case in the split
REF_T = 3000.0              # ref_scales.T -> Kelvin

def _latest(pattern):
    """Newest path matching an outputs/ glob (or None)."""
    hits = sorted(REPO.glob(pattern))
    return hits[-1] if hits else None

# Every random-path test-eval run is auto-discovered, keyed by run name
# (rigno_rp3, rigno_rp4, dpn_rp1, ... - latest timestamp per run), so different
# runs of the same family can be compared side by side. Add entries manually to
# pin a specific eval dir.
import re as _re
_RUN_RE = _re.compile(r"^(?P<run>(fno|dpn|rigno)_rp\d+)_test_\d{4}_\d{4}$")
MODEL_DIRS = {}
for _d in sorted(REPO.glob("outputs/*_test_*")):
    _m = _RUN_RE.match(_d.name)
    if _m:
        MODEL_DIRS[_m.group("run")] = _d      # sorted glob -> latest stamp wins
print("model eval dirs:", {k: str(v.name) for k, v in MODEL_DIRS.items()})


In [ ]:
def _find_eval_npz(model_dir):
    model_dir = Path(model_dir)
    pred = model_dir / "fno_eval" / "predictions.npz"
    gt = model_dir / "fno_eval" / "ground_truth.npz"
    if pred.exists() and gt.exists():
        return pred, gt
    ph = sorted(model_dir.rglob("predictions.npz")); gh = sorted(model_dir.rglob("ground_truth.npz"))
    if not ph or not gh:
        raise FileNotFoundError(f"predictions/ground_truth not found under {model_dir}")
    return ph[0], gh[0]

def _to_btcxyz(arr, t_per_case):
    # (B*T, C, X, Y, Z) -> (B, T, C, X, Y, Z) for time_as_batch eval output.
    arr = np.asarray(arr)
    if arr.ndim == 6:
        return arr
    if arr.ndim == 5:
        bt, c, x, y, z = arr.shape
        if bt % t_per_case != 0:
            raise ValueError(f"B*T={bt} not divisible by timesteps_per_case={t_per_case}")
        return arr.reshape(bt // t_per_case, t_per_case, c, x, y, z)
    raise ValueError(f"unexpected eval array ndim {arr.ndim}, shape {arr.shape}")

def load_model(model_dir, field="T"):
    pred_p, gt_p = _find_eval_npz(model_dir)
    pred = np.load(pred_p); gt = np.load(gt_p)
    p = _to_btcxyz(pred[field], TIMESTEPS_PER_CASE) * REF_T
    g = _to_btcxyz(gt[field], TIMESTEPS_PER_CASE) * REF_T
    return {"pred": p, "gt": g}

# Coordinates + laser paths from the sorted test NPZs (eval iterates cases in sorted order).
import sys
sys.path.insert(0, str(REPO / "src"))
from lpno.custom_utils import npz_datasource as nds

def load_coords_and_paths():
    files = sorted(f for f in TEST_DIR.glob("*.npz") if not f.name.startswith("var_"))
    d0 = np.load(files[0])
    x, y, z = d0["x"], d0["y"], d0["z"]
    paths, centres, names = [], [], []
    for f in files:
        d = np.load(f); t = d["time_s"]
        man = nds._load_manifest(f)
        xy = np.array(man["derived"]["laser_path_xy_m"]) if man else None
        c = nds._beam_centres_over_time(man, t) if man else None
        paths.append(xy); centres.append(c); names.append(f.stem)
    return x, y, z, paths, centres, names

X, Y, Z, PATHS, CENTRES, NAMES = load_coords_and_paths()
Z_UM = Z * 1e6
Z0 = int(np.argmin(np.abs(Z)))     # index nearest the z=0 surface
print(f"grid x,y,z = {X.size},{Y.size},{Z.size}; surface z=0 at index {Z0} ({Z_UM[Z0]:.0f} um)")
print(f"test cases: {len(NAMES)}")


In [ ]:
DATA = {}
for name, d in MODEL_DIRS.items():
    try:
        DATA[name] = load_model(d)
        print(f"loaded {name}: pred {DATA[name]['pred'].shape}")
    except Exception as e:
        print(f"skip {name}: {e}")
assert DATA, "No model eval outputs loaded -- set MODEL_DIRS to your eval run dirs."


In [ ]:
def show(model, case, t_idx, z_idx):
    d = DATA[model]
    g = d["gt"][case, t_idx, 0, :, :, z_idx]     # (X, Y)
    p = d["pred"][case, t_idx, 0, :, :, z_idx]
    vmin = float(min(g.min(), p.min())); vmax = float(max(g.max(), p.max()))
    ext = [X.min()*1e3, X.max()*1e3, Y.min()*1e3, Y.max()*1e3]   # mm
    fig, ax = plt.subplots(3, 1, figsize=(6.2, 11), constrained_layout=True)
    for a, arr, title, cmap, vlo, vhi in [
        (ax[0], g, "Ground truth  T [K]", "inferno", vmin, vmax),
        (ax[1], p, "Prediction  T [K]", "inferno", vmin, vmax),
        (ax[2], p - g, "Prediction - GT  [K]", "coolwarm", -max(1.0, abs(p-g).max()), max(1.0, abs(p-g).max())),
    ]:
        im = a.imshow(arr.T, origin="lower", extent=ext, cmap=cmap, aspect="auto")
        im.set_clim(vlo, vhi)
        fig.colorbar(im, ax=a, shrink=0.85)
        a.set_xlabel("x [mm]"); a.set_ylabel("y [mm]"); a.set_title(title)
        # overlay full laser path + instantaneous beam centre
        if PATHS[case] is not None:
            a.plot(PATHS[case][:, 0]*1e3, PATHS[case][:, 1]*1e3, "-", color="cyan", lw=0.8, alpha=0.7)
        if CENTRES[case] is not None:
            a.plot(CENTRES[case][t_idx, 0]*1e3, CENTRES[case][t_idx, 1]*1e3, "o", color="lime", ms=7, mfc="none", mew=2)
    fig.suptitle(f"{model}  |  {NAMES[case]}  |  t-index {t_idx}  |  z={Z_UM[z_idx]:.0f} um", fontsize=11)
    plt.show()

_model = widgets.Dropdown(options=list(DATA), description="model")
_case = widgets.IntSlider(min=0, max=len(NAMES)-1, value=0, description="case")
_t = widgets.IntSlider(min=0, max=TIMESTEPS_PER_CASE-1, value=TIMESTEPS_PER_CASE-1, description="time")
_z = widgets.IntSlider(min=0, max=Z.size-1, value=Z0, description="z")
widgets.interact(show, model=_model, case=_case, t_idx=_t, z_idx=_z);


## Cross-run comparison

One row of predictions (shared color scale with GT) and one row of error maps
(shared symmetric scale) for every discovered run. Failure modes that vanish in
volume-averaged metrics are visible here: dpn_rp1 paints the whole path warm
(the dose shortcut), dpn_rp2 adds checkerboard artifacts and overshoots at the
beam, fno_rp2 carries a broad warm bias below the track, rigno_rp3 shows
pixel-level decode speckle near the pool, rigno_rp4 is smooth but grows
coherent bias blobs (spurious warm spot at the path start, weak peak).

The two static plots below show *where* the error lives: it grows with frame
index (accumulated path -> harder thermal history; dpn_rp1's late-frame blowup
is the dose shortcut) and concentrates just below the z=0 surface (the gas
slices above are ~zero in both GT and prediction, which dilutes volume
metrics).

In [ ]:
def compare(case, t_idx, z_idx, err_clim):
    runs = list(DATA)
    g = DATA[runs[0]]["gt"][case, t_idx, 0, :, :, z_idx]
    n = len(runs) + 1
    fig, axes = plt.subplots(2, n, figsize=(2.9 * n, 6.2), constrained_layout=True)
    vmax = float(max(g.max(), max(DATA[r]["pred"][case, t_idx, 0, :, :, z_idx].max() for r in runs)))
    im = axes[0, 0].imshow(g.T, origin="lower", cmap="inferno", vmin=280, vmax=vmax, aspect="auto")
    axes[0, 0].set_title("GT", fontsize=9)
    fig.colorbar(im, ax=axes[0, 0], shrink=0.8)
    axes[1, 0].axis("off")
    for j, r in enumerate(runs, start=1):
        p = DATA[r]["pred"][case, t_idx, 0, :, :, z_idx]
        axes[0, j].imshow(p.T, origin="lower", cmap="inferno", vmin=280, vmax=vmax, aspect="auto")
        axes[0, j].set_title(r, fontsize=9)
        d = p - g
        im2 = axes[1, j].imshow(d.T, origin="lower", cmap="coolwarm", vmin=-err_clim, vmax=err_clim, aspect="auto")
        axes[1, j].set_title(f"err (slice MAE {np.abs(d).mean():.0f} K)", fontsize=8)
    fig.colorbar(im2, ax=axes[1, -1], shrink=0.8)
    for a in axes.ravel():
        a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f"{NAMES[case]}  |  t-index {t_idx}  |  z={Z_UM[z_idx]:.0f} um", fontsize=11)
    plt.show()

widgets.interact(
    compare,
    case=widgets.IntSlider(min=0, max=len(NAMES) - 1, value=0, description="case"),
    t_idx=widgets.IntSlider(min=0, max=TIMESTEPS_PER_CASE - 1, value=TIMESTEPS_PER_CASE - 1, description="time"),
    z_idx=widgets.IntSlider(min=0, max=Z.size - 1, value=Z0, description="z"),
    err_clim=widgets.IntSlider(min=50, max=500, step=25, value=150, description="err clim [K]"),
);

In [ ]:
# Error structure: MAE vs frame index (left) and vs depth (right), per run.
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6), constrained_layout=True)
for r in DATA:
    e = np.abs(DATA[r]["pred"] - DATA[r]["gt"])
    a1.plot(e.mean(axis=(0, 2, 3, 4, 5)), label=r)
    a2.plot(Z_UM, e.mean(axis=(0, 1, 2, 3, 4)), label=r)
a1.set_xlabel("frame index"); a1.set_ylabel("MAE [K]"); a1.set_title("error vs time")
a2.axvline(0.0, color="k", lw=0.6, ls="--")
a2.set_xlabel("z [um]"); a2.set_title("error vs depth (dashed: z=0 surface)")
a1.legend(fontsize=7)
plt.show()

## Straight-line baseline viewer

Same top-bottom layout for the straight-line study the baseline numbers come from
(FNO `fno_oracle` and RIGNO config M). One semantic difference: that study evaluates a
**single moving-frame, 30-frame time-averaged field per case** (the frame travels with
the beam and the last 30 exported steps are averaged), so there is no time axis — the
sliders are case and z only. Grid [61,40,30] at the native 10 µm (not strided); 9 test
cases varying laser power (50–150 W) and scan speed (0.26–0.71 m/s). The cyan line is
the weld centerline y=0; the beam sits at a fixed position in this frame.

In [ ]:
# Straight-line baseline evals: one moving-frame, 30-frame time-averaged field per case.
SL_MODEL_DIRS = {k: v for k, v in {
    "FNO (straight)":   _latest("outputs/fno_oracle_test_*"),
    "RIGNO (straight)": _latest("outputs/rigno_M_test_*"),
}.items() if v is not None}
SL_DATA_DIR = REPO / "data/flow3D/original_straight_path_data/test"

SL_DATA = {}
for name, d in SL_MODEL_DIRS.items():
    pred_p, gt_p = _find_eval_npz(d)
    SL_DATA[name] = {"pred": np.load(pred_p)["T"] * REF_T,   # (9, 1, 61, 40, 30)
                     "gt":   np.load(gt_p)["T"] * REF_T}
print("straight-line evals:", {k: v["pred"].shape for k, v in SL_DATA.items()})

# The consolidated cache holds the processed moving-frame coordinates on the eval grid.
_sl = np.load(SL_DATA_DIR / "var_1_time_steps30_avg30.npz")
SLX = _sl["x"][0, 0, :, 0, 0]; SLY = _sl["y"][0, 0, 0, :, 0]; SLZ = _sl["z"][0, 0, 0, 0, :]
SL_NAMES = sorted(p.name for p in SL_DATA_DIR.iterdir() if p.is_dir() and not p.name.startswith("."))
SL_Z0 = int(np.argmin(np.abs(SLZ)))
print(f"straight grid x,y,z = {SLX.size},{SLY.size},{SLZ.size}; surface at z-index {SL_Z0} ({SLZ[SL_Z0]*1e6:.0f} um)")

def show_straight(model, case, z_idx):
    d = SL_DATA[model]
    g = d["gt"][case, 0, :, :, z_idx]     # (X, Y)
    p = d["pred"][case, 0, :, :, z_idx]
    vmin = float(min(g.min(), p.min())); vmax = float(max(g.max(), p.max()))
    ext = [SLX.min()*1e3, SLX.max()*1e3, SLY.min()*1e3, SLY.max()*1e3]   # mm
    fig, ax = plt.subplots(3, 1, figsize=(6.2, 11), constrained_layout=True)
    for a, arr, title, cmap, vlo, vhi in [
        (ax[0], g, "Ground truth  T [K]", "inferno", vmin, vmax),
        (ax[1], p, "Prediction  T [K]", "inferno", vmin, vmax),
        (ax[2], p - g, "Prediction - GT  [K]", "coolwarm", -max(1.0, abs(p-g).max()), max(1.0, abs(p-g).max())),
    ]:
        im = a.imshow(arr.T, origin="lower", extent=ext, cmap=cmap, aspect="auto")
        im.set_clim(vlo, vhi)
        fig.colorbar(im, ax=a, shrink=0.85)
        a.set_xlabel("x [mm] (moving frame)"); a.set_ylabel("y [mm]"); a.set_title(title)
        a.axhline(0.0, color="cyan", lw=0.8, alpha=0.7)   # weld centerline
    fig.suptitle(f"{model}  |  {SL_NAMES[case]}  |  30-frame moving-frame average  |  z={SLZ[z_idx]*1e6:.0f} um",
                 fontsize=11)
    plt.show()

_sm = widgets.Dropdown(options=list(SL_DATA), description="model")
_sc = widgets.IntSlider(min=0, max=len(SL_NAMES)-1, value=0, description="case")
_sz = widgets.IntSlider(min=0, max=SLZ.size-1, value=SL_Z0, description="z")
widgets.interact(show_straight, model=_sm, case=_sc, z_idx=_sz);

## Test-set metrics and comparison with the straight-line baseline

Metrics are read from each eval's `fno_eval/model_errors.txt` (normalized units;
multiply T by 3000 for Kelvin). The straight-line baseline column is the FNO/RIGNO
straight-line result (moving-frame, constant-speed x-scan).

In [ ]:
# Region metrics computed uniformly from the eval arrays for every discovered
# run (older runs' model_errors.txt lack the substrate_*/surface_* stats the
# eval pipeline now emits; recomputing keeps the comparison consistent).
# substrate = z<=0 (excludes the ~zero-error gas slices that dilute volume
# averages by ~25%); surface = the z-slice nearest 0 from below.
SUB = Z <= 1e-9
print(f"{'run':10s} | {'vol MAE':>8s} {'vol RMSE':>9s} | {'sub MAE':>8s} {'sub RMSE':>9s} | {'surf MAE':>9s} {'surf RMSE':>10s}   [K]")
for name, d in DATA.items():
    e = d["pred"] - d["gt"]
    v = [np.abs(e).mean(), np.sqrt((e ** 2).mean()),
         np.abs(e[..., SUB]).mean(), np.sqrt((e[..., SUB] ** 2).mean()),
         np.abs(e[..., Z0]).mean(), np.sqrt((e[..., Z0] ** 2).mean())]
    print(f"{name:10s} | {v[0]:8.1f} {v[1]:9.1f} | {v[2]:8.1f} {v[3]:9.1f} | {v[4]:9.1f} {v[5]:10.1f}")

# Straight-line baselines:
# - moving-frame, 30-frame time-averaged evals (fno_oracle / rigno_M): volume
#   MAE/RMSE 0.00376/0.00844 (FNO) and 0.00336/0.00733 (RIGNO) in normalized T
#   (x3000 for Kelvin). Time-averaging smooths keyhole peaks, so these are NOT
#   apples-to-apples with the per-frame numbers above.
# - per-frame lab-frame baselines on the identical pipeline (rigno_sl1 /
#   fno_sl1, data/flow3D/resampled_straight_path_data) are training; their evals land as
#   outputs/{rigno,fno}_sl1_test_* and should be compared with these region
#   metrics.
print("\nsurface-slice errors of the moving-frame straight-line evals (Kelvin):")
for name in SL_DATA:
    p, g = SL_DATA[name]["pred"][..., SL_Z0], SL_DATA[name]["gt"][..., SL_Z0]
    print(f"{name:24s} surf MAE {np.abs(p - g).mean():6.1f} K   surf RMSE {np.sqrt(((p - g) ** 2).mean()):6.1f} K")

## Findings

**Setup.** 60 FLOW-3D cases (50 random-walk + 10 sinusoidal laser paths, TiAl64, 150 W,
0.8 m/s, 1.5 mm path in a 1.5×1.0×0.3 mm substrate), split 36/12/12 (train/val/test)
stratified by path family; 20 frames per case, grid strided to [75,50,40] (20 µm in x,y).
Lab frame, no moving reference frame; time inference via `time_as_batch` (each frame a
sample, `t_hat` a query input). The laser trajectory enters as explicit input channels
derived from each case's manifest: instantaneous beam centre (`beam_x`,`beam_y`), Gaussian
footprint (`beam_dist`), and accumulated exposure (`path_dose` — the thermal-memory
channel). DeepONet routes `path_dose` to its trunk (a spatial field cannot sit in the
branch); the others take all four as grid channels.

**Results (normalized by T_ref=3000 K; ×3000 for Kelvin).** Test set. T rel-err is
`mean(|T_pred − T_true|) / mean(T_true)` — the MAE as a fraction of the mean field
temperature, dimensionless and comparable across grids/normalizations:

| model | T MAE | T RMSE | T rel-err | melt-region IoU |
|---|---|---|---|---|
| RIGNO (random-path) | **0.00400** (12.0 K) | **0.01350** (40.5 K) | **0.0117** | 0.9997 |
| FNO 18k (random-path) | 0.00531 (15.9 K) | 0.01730 (51.9 K) | 0.0155 | 0.9997 |
| DeepONet rp1 (random-path) | 0.00762 (22.9 K) | 0.02669 (80.1 K) | 0.0222 | 0.9991 |
| DeepONet rp2 (+beam_dist in trunk) | 0.00665 (20.0 K) | 0.02103 (63.1 K) | 0.0194 | 0.9995 |
| RIGNO rp4 (edge-mask 0.1 + grad-loss 1.0) | 0.00513 (15.4 K) | 0.01888 (56.7 K) | 0.0150 | 0.9995 |
| FNO straight-line baseline | 0.00376 (11.3 K) | 0.00844 (25.3 K) | 0.0190 | 0.9994 |
| RIGNO straight-line (config M) | 0.00336 (10.1 K) | 0.00733 (22.0 K) | 0.0170 | 0.9994 |

Validation set (the comparability criterion): RIGNO **0.00411 / 0.01438** and FNO
**0.00508 / 0.01674** vs the straight-line FNO baseline **0.00453 / 0.00967** — random-path
RIGNO *beats* the baseline's val MAE and halves its rel-err (0.0120 vs 0.0228); FNO is
within 12% on MAE. RMSE sits at 1.5–1.7× the baseline.

**Interpretation.**
1. *Arbitrary-path conditioning works.* With the trajectory passed explicitly, the models
   localize heating along curved, self-crossing paths they never saw — melt-region IoU
   ≥0.999 on all three, and MAE at or better than the straight-line baseline.
2. *RIGNO > FNO > DeepONet*, the same ordering as the straight-line study. On rel-err the
   random-path RIGNO (0.0117) beats even the straight-line RIGNO (0.0170).
3. *Volume metrics are correct but diluted.* The `model_errors.txt` numbers reproduce
   exactly when recomputed from the raw eval arrays, but they average over the whole
   volume: the 10 gas-layer z-slices carry ~zero error (masked in both GT and prediction)
   and the deep substrate is easy, so the z≈0 surface — where the viewer shows all the
   artifacts — runs ~3–4× worse than the volume average (surface MAE: RIGNO 35.6 K,
   FNO 57.8 K, DeepONet 76.3 K vs volume 12.0/15.9/22.9 K). The ranking is unchanged;
   FNO's visible surface artifacts are real and are why it trails RIGNO there.
4. *Both GT and prediction share the same strided 20 µm mesh*, so mesh coarseness cannot
   explain prediction-vs-GT differences in the viewer: the GT panel looks smooth because
   the physical field is smooth. RIGNO's patchy blocks are its regional-graph decode
   footprint; its weakened conduction halo (and FNO's cold blotches) are model error —
   the smooth low-amplitude far field contributes little to the training loss.
5. *DeepONet's warm-snake artifact is a `path_dose` shortcut.* It paints the whole path
   at ~1600–2000 K where GT has cooled below 900 K: at the final frame, on the 8 earliest
   beam positions of test case `tial64_rand_batch_06`, GT reads 589–852 K, FNO/RIGNO track
   within ~110 K, DeepONet predicts 971–1660 K. Cause: its output is a rank-limited
   branch⊗trunk product; the only *spatial* conditioning it gets is `path_dose` in the
   trunk (it cannot take the `beam_dist` footprint grid channel, and beam_x/beam_y enter
   only as branch scalars), and synthesizing a sharp bump at an arbitrary scalar-specified
   position is exactly what a low-rank basis is bad at — so it leans on the dose field,
   which marks where the beam *has been* with no cooling. Lever: route `beam_dist` to the
   trunk too (`custom.deeponet_trunk_extra_keys: [path_dose, beam_dist]`) or add a
   time-decayed dose channel. *(beam_dist was routed to the trunk in dpn_rp2 — see the
   2026-07-06 update below: overall error improved but the shortcut persists.)*
6. *The remaining RMSE gap is keyhole peaks, not path errors.* GT temperature reaches
   ~3888 K in the vapor depression; all models saturate near the boiling fill (~3123 K),
   and the error histogram is dominated by those cells. Two structural handicaps vs the
   baseline numbers: this eval scores instantaneous per-frame fields (the baseline scores
   30-frame time-averaged moving-frame fields, which smooths peaks), and the grid here is
   2× coarser in x,y (memory constraint of per-frame training).
7. *Per-frame liquid-fraction IoU (~0.65–0.69 for grid models) is not comparable to the
   baseline's 0.93*: the fl>0.5 band is ~50 K wide, so instantaneous-frame timing errors
   dominate; the DeepONet fl numbers are additionally an eval artifact (fl derived from a
   predicted T read 1 across the gas layer; since fixed).
8. Training gotchas hit and fixed: FNO diverged at the shipped lr 1e-4 (fixed: 3e-5 +
   batch 4; 18k steps with slower decay halved v1's error); RIGNO NaN'd deterministically
   in the relobralo loss aggregator on this per-frame data (fixed with `loss: sum`).

**Next levers** (if the RMSE gap matters): train at the native 10 µm grid with a lazy
loader, add the vapor region to the loss weighting, or a log-T target transform to
de-emphasize the saturated fill and resolve superheat.

---

**Update 2026-07-06** (rp4 / dpn_rp2 rounds; see the cross-run comparison cells above):

- *RIGNO rp4 traded speckle for bias and lost.* Edge masking 0.1 + gradient loss 1.0
  removed rp3's pixel-level decode speckle (the error field is smooth) but added large
  coherent bias blobs: a spurious ~+150 K warm spot at the path start, an over-warm band
  trailing the pool, and an under-predicted peak. Test vol MAE 12.0 -> 15.4 K, surface
  35.6 -> 50.1 K, and even the gradient-mismatch RMSE it targeted worsened (27.6 -> 37.6
  K/cell). Config defaults were reverted to the rp3 recipe; ablations `rigno_rp5`
  (masking 0.1 only) and `rigno_rp6` (masking 0.1 + gradient loss 0.1) are queued to
  attribute the smoothing vs the bias.
- *DeepONet rp2 (beam_dist in the trunk) helps but does not fix the dose shortcut.* Vol
  MAE 22.9 -> 20.0 K, RMSE 80 -> 63 K, and the late-frame error growth flattens (frame-19
  MAE 43.5 -> 35.0 K). But on early-path points that GT cooled below 900 K by the last
  frame (GT mean 715 K), rp1 predicted +442 K too warm and rp2 +418 K: better hot-zone
  localization, same missing physics. No input carries per-point *elapsed cooling time*
  (path_dose is cumulative, without timestamps). dpn_rp3 candidate: a time-decayed dose
  (recency) channel in the trunk.
- *Region metrics are now standard.* Eval emits substrate_*/surface_* stats
  (postprocessing.region_error_stats); the metrics cell above recomputes them uniformly
  for old runs. Volume numbers on this grid are diluted ~25% by the gas slices.
- *Per-frame straight-line RIGNO baseline landed* (`rigno_sl1`, rp3-matched recipe,
  data/flow3D/resampled_straight_path_data): test substrate MAE 15.0 K / RMSE 39.3 K, surface MAE
  29.3 K / RMSE 81.6 K (straight grids end at z=0, so substrate = volume). Against
  rigno_rp3 on random paths (substrate 15.7 K, surface 35.6 K): arbitrary-path
  conditioning costs ~4% in the substrate and ~20% at the surface. The per-frame
  straight task is much harder than the old moving-frame time-averaged eval (T rel-err
  0.034 vs rigno_M's 0.017) — time-averaging flattered the old baseline numbers. The
  FNO/DeepONet per-frame baselines run in a parallel session (fno_straight_tb,
  dpn_straight_tb).
- *Training-speed changes* (for future runs): fp16 AMP is on (1.53x on RIGNO: 272 vs 420
  ms/it; needed a missing physicsnemo_ext op-registration import), CUDA graphs stay off
  (WarmupNode's step.item() sync breaks capture), training records are scalar-only
  (validator plots/file dumps opt-in via custom.validator_plots), DeepONet validation
  batch 10k -> 100k, early stopping on l2_relative_error_T, DeepONet steps 15k -> 12k
  (rp1 plateaued ~9-12k).

## Post-hoc corrector: rp3 uncorrected vs corrected

The RIGNO tuning campaign (rp4 edge-masking+grad, rp5 grad-only, rp7 k_r2p,
rp8 subsample) all lost to plain rp3, because they target the ~24% blocky part
of the error while the dominant ~76% is a smooth spatial bias (see the residual
decomposition). So instead we trained a **post-hoc corrector**: map rp3's
predicted field -> GT temperature. A pointwise skip-MLP (FC-NN) and a small
spatial 3D-FNO, trained on rp3's val+train predictions (48 cases), evaluated on
the held-out **test** set. The spatial FNO wins decisively — it can see and
subtract the coherent bias a pointwise map can't.

`outputs/corrector_artifacts/corrector_preds_aug.npz` holds `base` (rp3),
`mlp`, `fno`, and `gt` test-set fields (240 frames = 12 cases x 20).

In [ ]:
import numpy as np
_S = 3000.0
_cp = np.load(REPO / "outputs/corrector_artifacts/corrector_preds_aug.npz")
CB, CM, CF, CG = (_cp[k].reshape(12, 20, 75, 50, 40) for k in ("base", "mlp", "fno", "gt"))
_zsub = slice(0, 30)   # substrate z<=0; surface = index 29 (-5 um)
def _rm(pred):
    e = (pred - CG) * _S
    return (np.abs(e).mean(), np.abs(e[..., _zsub]).mean(), np.abs(e[..., 29]).mean())
print(f"{'model':22s} {'vol MAE':>8s} {'sub MAE':>8s} {'surf MAE':>9s}   (K, test set)")
print("-" * 52)
for _name, _p in [("rp3 uncorrected", CB), ("+ FC-NN (pointwise)", CM), ("+ FNO (spatial)", CF)]:
    v, s, u = _rm(_p)
    _d = "" if _p is CB else f"  ({100*(1 - v/_rm(CB)[0]):+.0f}% vol)"
    print(f"{_name:22s} {v:8.1f} {s:8.1f} {u:9.1f}{_d}")

In [ ]:
# rp3 error vs corrected error: the difference the FNO corrector removes.
def show_correction(case, t_idx, z_idx):
    g = CG[case, t_idx, :, :, z_idx]
    eb = CB[case, t_idx, :, :, z_idx] - g
    ef = CF[case, t_idx, :, :, z_idx] - g
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
    vmax = float(g.max()) * 3000
    im0 = ax[0].imshow((g * 3000).T, origin="lower", cmap="inferno", vmin=280, vmax=vmax, aspect="auto")
    ax[0].set_title("GT  T [K]", fontsize=10)
    fig.colorbar(im0, ax=ax[0], shrink=0.8)
    for a, d, t in [(ax[1], eb, "rp3 - GT"), (ax[2], ef, "rp3 + FNO corrector - GT")]:
        im = a.imshow((d * 3000).T, origin="lower", cmap="coolwarm", vmin=-150, vmax=150, aspect="auto")
        a.set_title(f"{t}  (MAE {np.abs(d*3000).mean():.0f} K)", fontsize=9)
    fig.colorbar(im, ax=ax[2], shrink=0.8)
    for a in ax: a.set_xticks([]); a.set_yticks([])
    plt.show()

widgets.interact(
    show_correction,
    case=widgets.IntSlider(min=0, max=11, value=0, description="case"),
    t_idx=widgets.IntSlider(min=0, max=19, value=19, description="time"),
    z_idx=widgets.IntSlider(min=0, max=39, value=29, description="z"),
)

# per-case volume MAE, rp3 vs corrected
_cb = np.abs((CB - CG) * _S).mean(axis=(1, 2, 3, 4))
_cf = np.abs((CF - CG) * _S).mean(axis=(1, 2, 3, 4))
fig, axb = plt.subplots(figsize=(9, 3.2))
_x = np.arange(12)
axb.bar(_x - 0.2, _cb, 0.4, label="rp3 uncorrected", color="#c44")
axb.bar(_x + 0.2, _cf, 0.4, label="rp3 + FNO corrector", color="#48c")
axb.set_xlabel("test case"); axb.set_ylabel("volume MAE [K]")
axb.set_title(f"FNO corrector: {np.abs((CB-CG)*_S).mean():.1f} -> {np.abs((CF-CG)*_S).mean():.1f} K vol MAE (held-out test)")
axb.legend(fontsize=8); axb.set_xticks(_x)
plt.tight_layout(); plt.show()

## End-to-end refinement: RIGNO + residual FNO head vs rp3

The corrector showed a spatial FNO can cut rp3's smooth-bias error post-hoc
(augmented/leaked +31% on test). `model_family=rigno_refined`
(`RIGNORefinedArch`) wires that FNO as a **residual head trained end-to-end**
(leakage-free): `refined = rigno(x) + head(rigno(x))`, head zero-init so it
starts as pure RIGNO. Variants: base trained from scratch jointly; base
**warm-started** from rp3's weights; and warm-start with a separate, higher head
LR (`custom.head_lr`) so the head can learn the correction on a protected base.
Region metrics on the held-out **test** set, Kelvin. (Rows auto-appear as evals
land.)

In [ ]:
# rp3 vs the end-to-end refinement variants (whichever eval dirs exist).
_REFINED_RUNS = {
    "rp3 (baseline)":              "outputs/rigno_rp3_test_*",
    "refined from-scratch (20k)":  "outputs/rigno_refined_20k_test_*",
    "refined warm-start (16k)":    "outputs/rigno_refined_ws_test*",
    "refined ws + head-LR (16k)":  "outputs/rigno_refined_ws_hlr_test_*",
    "refined ws + head-LR (36k)":  "outputs/rigno_refined_ws_hlr36k_test_*",
}
_zsub = slice(0, 30)   # substrate z<=0; surface = index 29 (-5 um)
def _load_refined_T(glob_pat):
    # newest matching eval dir that actually has predictions (skip failed evals)
    for d in sorted(REPO.glob(glob_pat), reverse=True):
        pf = d / "fno_eval" / "predictions.npz"
        if pf.exists():
            P = np.load(pf)["T"].astype(np.float64) * REF_T
            G = np.load(d / "fno_eval" / "ground_truth.npz")["T"].astype(np.float64) * REF_T
            return P, G
    return None
REFINED = {}
print(f"{'model':28s} {'vol MAE':>8s} {'vol RMSE':>9s} {'sub MAE':>8s} {'surf MAE':>9s}   (K, held-out test)")
print("-" * 68)
_rp3_vol = None
for _name, _g in _REFINED_RUNS.items():
    _r = _load_refined_T(_g)
    if _r is None:
        continue
    P, G = _r; REFINED[_name] = (P, G); e = P - G
    vol = np.abs(e).mean()
    if _rp3_vol is None: _rp3_vol = vol
    _d = "" if _name.startswith("rp3") else f"  ({100*(1 - vol/_rp3_vol):+.0f}% vs rp3)"
    print(f"{_name:28s} {vol:8.1f} {np.sqrt((e**2).mean()):9.1f} "
          f"{np.abs(e[..., _zsub]).mean():8.1f} {np.abs(e[..., 29]).mean():9.1f}{_d}")

In [ ]:
# rp3 error vs the best available refined variant (last non-rp3 row loaded).
_variants = [k for k in REFINED if not k.startswith("rp3")]
if REFINED and _variants:
    _best = _variants[-1]
    _Prp3, _G = (v.reshape(12, 20, 75, 50, 40) for v in REFINED["rp3 (baseline)"])
    _Pref, _  = (v.reshape(12, 20, 75, 50, 40) for v in REFINED[_best])
    def show_refined(case, t_idx, z_idx):
        g = _G[case, t_idx, :, :, z_idx]
        eb = _Prp3[case, t_idx, :, :, z_idx] - g
        er = _Pref[case, t_idx, :, :, z_idx] - g
        fig, ax = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
        im0 = ax[0].imshow(g.T, origin="lower", cmap="inferno", vmin=280, vmax=float(g.max()), aspect="auto")
        ax[0].set_title("GT  T [K]", fontsize=10); fig.colorbar(im0, ax=ax[0], shrink=0.8)
        for a, d, t in [(ax[1], eb, "rp3 - GT"), (ax[2], er, f"{_best} - GT")]:
            im = a.imshow(d.T, origin="lower", cmap="coolwarm", vmin=-150, vmax=150, aspect="auto")
            a.set_title(f"{t}  (MAE {np.abs(d).mean():.0f} K)", fontsize=9)
        fig.colorbar(im, ax=ax[2], shrink=0.8)
        for a in ax: a.set_xticks([]); a.set_yticks([])
        plt.show()
    widgets.interact(show_refined,
                     case=widgets.IntSlider(min=0, max=11, value=0, description="case"),
                     t_idx=widgets.IntSlider(min=0, max=19, value=19, description="time"),
                     z_idx=widgets.IntSlider(min=0, max=39, value=29, description="z"))
else:
    print("no refined eval yet")

In [ ]:
# Raw predicted T fields side by side (GT / rp3 / best refined), same colormap
# and scale -- error maps can hide spurious diffusion, smeared gradients, or a
# missing/extra hot spot that the raw field makes obvious.
if REFINED and _variants:
    def show_raw(case, t_idx, z_idx):
        g  = _G[case, t_idx, :, :, z_idx]
        pr = _Prp3[case, t_idx, :, :, z_idx]
        pf = _Pref[case, t_idx, :, :, z_idx]
        vmax = float(max(g.max(), pr.max(), pf.max()))
        fig, ax = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
        for a, d, t in [(ax[0], g, "GT"), (ax[1], pr, "rp3"), (ax[2], pf, _best)]:
            im = a.imshow(d.T, origin="lower", cmap="inferno", vmin=280, vmax=vmax, aspect="auto")
            a.set_title(f"{t}   T [K]  (peak {d.max():.0f})", fontsize=9)
            a.set_xticks([]); a.set_yticks([])
        fig.colorbar(im, ax=ax[2], shrink=0.8)
        plt.show()
    widgets.interact(show_raw,
                     case=widgets.IntSlider(min=0, max=11, value=0, description="case"),
                     t_idx=widgets.IntSlider(min=0, max=19, value=19, description="time"),
                     z_idx=widgets.IntSlider(min=0, max=39, value=29, description="z"))
else:
    print("no refined eval yet")